In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

In [2]:
from sklearn.model_selection import train_test_split

DATA_DIR = Path("../data/interim")

features = pd.read_csv(
    DATA_DIR / "customer_features.csv"
)

labels = pd.read_csv(
    DATA_DIR / "churn_labels.csv"
)

model_data = features.merge(
    labels,
    on="user_id",
    how="inner",
    validate="one_to_one",
)

TARGET = "churn_like_label"
ID_COLUMN = "user_id"

X = model_data.drop(
    columns=[ID_COLUMN, TARGET]
)

y = model_data[TARGET].astype("int8")

In [3]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (144329, 23)
Validation: (30928, 23)
Test: (30928, 23)


In [4]:
dummy_model = DummyClassifier(
    strategy="most_frequent"
)

dummy_model.fit(
    X_train,
    y_train
)

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](2,)","[0.79,0.21]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[int8](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](23,)","['total_orders','max_order_number','avg_days_between_orders',..., 'dominant_aisle_share','inactivity_gap','inactivity_ratio']"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,2
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,23
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [5]:
y_val_pred = dummy_model.predict(X_val)

In [6]:
baseline_results = {
    "Accuracy": accuracy_score(y_val, y_val_pred),
    "Precision": precision_score(
        y_val,
        y_val_pred,
        zero_division=0,
    ),
    "Recall": recall_score(
        y_val,
        y_val_pred,
        zero_division=0,
    ),
    "F1": f1_score(
        y_val,
        y_val_pred,
        zero_division=0,
    ),
}

baseline_results

{'Accuracy': 0.7906751163993792, 'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0}

In [7]:
cm = confusion_matrix(
    y_val,
    y_val_pred
)

cm

array([[24454,     0],
       [ 6474,     0]])

## Candidate Model Training

Candidate models are trained using preprocessing pipelines appropriate to their algorithmic requirements.

Logistic Regression requires numerical features on a comparable scale, so median imputation and standardization are included inside the model pipeline.

The validation set is used for initial model comparison, while the test set remains reserved for final evaluation.

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [9]:
logistic_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        ),
    ]
)

In [10]:
logistic_pipeline.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int8](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](23,)","['total_orders','max_order_number','avg_days_between_orders',..., 'dominant_aisle_share','inactivity_gap','inactivity_ratio']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,23
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace 

In [11]:
y_val_pred_lr = logistic_pipeline.predict(X_val)
y_val_prob_lr = logistic_pipeline.predict_proba(X_val)[:, 1]

In [12]:
logistic_results = {
    "Accuracy": accuracy_score(
        y_val,
        y_val_pred_lr
    ),
    "Precision": precision_score(
        y_val,
        y_val_pred_lr,
        zero_division=0
    ),
    "Recall": recall_score(
        y_val,
        y_val_pred_lr,
        zero_division=0
    ),
    "F1": f1_score(
        y_val,
        y_val_pred_lr,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_val,
        y_val_prob_lr
    ),
    "PR-AUC": average_precision_score(
        y_val,
        y_val_prob_lr
    ),
}

logistic_results

{'Accuracy': 0.7980794102431453,
 'Precision': 0.5406460773872914,
 'Recall': 0.23524868705591598,
 'F1': 0.32784415025293295,
 'ROC-AUC': 0.8113277009744535,
 'PR-AUC': 0.47235855949459976}

In [13]:
from sklearn.ensemble import RandomForestClassifier

In [14]:
random_forest_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
                class_weight="balanced"
            )
        ),
    ]
)

In [15]:
random_forest_pipeline.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int8](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](23,)","['total_orders','max_order_number','avg_days_between_orders',..., 'dominant_aisle_share','inactivity_gap','inactivity_ratio']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,23
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace allocc

In [16]:
y_val_pred_rf = random_forest_pipeline.predict(X_val)

y_val_prob_rf = (
    random_forest_pipeline
    .predict_proba(X_val)[:, 1]
)

In [17]:
random_forest_results = {
    "Accuracy": accuracy_score(
        y_val,
        y_val_pred_rf,
    ),
    "Precision": precision_score(
        y_val,
        y_val_pred_rf,
        zero_division=0,
    ),
    "Recall": recall_score(
        y_val,
        y_val_pred_rf,
        zero_division=0,
    ),
    "F1": f1_score(
        y_val,
        y_val_pred_rf,
        zero_division=0,
    ),
    "ROC-AUC": roc_auc_score(
        y_val,
        y_val_prob_rf,
    ),
    "PR-AUC": average_precision_score(
        y_val,
        y_val_prob_rf,
    ),
}

random_forest_results

{'Accuracy': 0.7608316088980859,
 'Precision': 0.44905618721713214,
 'Recall': 0.6283595922150139,
 'F1': 0.5237880641215477,
 'ROC-AUC': 0.8301931515152847,
 'PR-AUC': 0.47662745764572345}

In [24]:
model_results = pd.DataFrame(
    [
        {
            "Model": "Dummy Baseline",
            **baseline_results,
        },
        {
            "Model": "Logistic Regression",
            **logistic_results,
        },
        {
            "Model": "Random Forest",
            **random_forest_results,
        },
        {
            "Model": "Extra Trees",
            **extra_trees_results,
        },
    ]
)

model_results

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Dummy Baseline,0.790675,0.000000,0.000000,0.000000,NaN,NaN
1,Logistic Regression,0.798079,0.540646,0.235249,0.327844,0.811328,0.472359
2,Random Forest,0.760832,0.449056,0.628360,0.523788,0.830193,0.476627
3,Extra Trees,0.796042,0.531344,0.217331,0.308485,0.828035,0.478166


## Extra Trees

In [19]:
from sklearn.ensemble import ExtraTreesClassifier

In [20]:
extra_trees_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            ExtraTreesClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
                class_weight="balanced"
            )
        ),
    ]
)

In [21]:
extra_trees_pipeline.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int8](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](23,)","['total_orders','max_order_number','avg_days_between_orders',..., 'dominant_aisle_share','inactivity_gap','inactivity_ratio']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,23
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace allocc

In [22]:
y_val_pred_et = extra_trees_pipeline.predict(X_val)

y_val_prob_et = (
    extra_trees_pipeline
    .predict_proba(X_val)[:, 1]
)

In [23]:
extra_trees_results = {
    "Accuracy": accuracy_score(
        y_val,
        y_val_pred_et,
    ),
    "Precision": precision_score(
        y_val,
        y_val_pred_et,
        zero_division=0,
    ),
    "Recall": recall_score(
        y_val,
        y_val_pred_et,
        zero_division=0,
    ),
    "F1": f1_score(
        y_val,
        y_val_pred_et,
        zero_division=0,
    ),
    "ROC-AUC": roc_auc_score(
        y_val,
        y_val_prob_et,
    ),
    "PR-AUC": average_precision_score(
        y_val,
        y_val_prob_et,
    ),
}

extra_trees_results

{'Accuracy': 0.7960424211070875,
 'Precision': 0.5313444108761329,
 'Recall': 0.21733086190917517,
 'F1': 0.30848498136373603,
 'ROC-AUC': 0.8280354780345912,
 'PR-AUC': 0.4781660996551349}

## HistGradientBoosting

In [25]:
from sklearn.ensemble import HistGradientBoostingClassifier

In [26]:
hist_gb_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            HistGradientBoostingClassifier(
                max_iter=300,
                random_state=42
            )
        ),
    ]
)

In [27]:
hist_gb_pipeline.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int8](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](23,)","['total_orders','max_order_number','avg_days_between_orders',..., 'dominant_aisle_share','inactivity_gap','inactivity_ratio']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,23
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace allocc

In [28]:
y_val_pred_hgb = hist_gb_pipeline.predict(X_val)

y_val_prob_hgb = (
    hist_gb_pipeline
    .predict_proba(X_val)[:, 1]
)

In [29]:
hist_gb_results = {
    "Accuracy": accuracy_score(
        y_val,
        y_val_pred_hgb,
    ),
    "Precision": precision_score(
        y_val,
        y_val_pred_hgb,
        zero_division=0,
    ),
    "Recall": recall_score(
        y_val,
        y_val_pred_hgb,
        zero_division=0,
    ),
    "F1": f1_score(
        y_val,
        y_val_pred_hgb,
        zero_division=0,
    ),
    "ROC-AUC": roc_auc_score(
        y_val,
        y_val_prob_hgb,
    ),
    "PR-AUC": average_precision_score(
        y_val,
        y_val_prob_hgb,
    ),
}

hist_gb_results

{'Accuracy': 0.7994374030005174,
 'Precision': 0.5581295581295581,
 'Recall': 0.20095767686129132,
 'F1': 0.2955139125496877,
 'ROC-AUC': 0.8375694554299135,
 'PR-AUC': 0.5022704442275266}

## XGBoost

In [32]:
from xgboost import XGBClassifier

In [33]:
xgb_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            XGBClassifier(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1,
            )
        ),
    ]
)

In [34]:
xgb_pipeline.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](23,)","['total_orders','max_order_number','avg_days_between_orders',..., 'dominant_aisle_share','inactivity_gap','inactivity_ratio']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,23
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloc

In [35]:
y_val_pred_xgb = xgb_pipeline.predict(X_val)

y_val_prob_xgb = (
    xgb_pipeline
    .predict_proba(X_val)[:, 1]
)

In [36]:
xgb_results = {
    "Accuracy": accuracy_score(
        y_val,
        y_val_pred_xgb,
    ),
    "Precision": precision_score(
        y_val,
        y_val_pred_xgb,
        zero_division=0,
    ),
    "Recall": recall_score(
        y_val,
        y_val_pred_xgb,
        zero_division=0,
    ),
    "F1": f1_score(
        y_val,
        y_val_pred_xgb,
        zero_division=0,
    ),
    "ROC-AUC": roc_auc_score(
        y_val,
        y_val_prob_xgb,
    ),
    "PR-AUC": average_precision_score(
        y_val,
        y_val_prob_xgb,
    ),
}

xgb_results

{'Accuracy': 0.8002457320227625,
 'Precision': 0.5612076095947064,
 'Recall': 0.20960766141489032,
 'F1': 0.30521817363922626,
 'ROC-AUC': 0.8380242380522966,
 'PR-AUC': 0.502038457510624}

In [37]:
model_results = pd.DataFrame(
    [
        {"Model": "Dummy Baseline", **baseline_results},
        {"Model": "Logistic Regression", **logistic_results},
        {"Model": "Random Forest", **random_forest_results},
        {"Model": "Extra Trees", **extra_trees_results},
        {"Model": "HistGradientBoosting", **hist_gb_results},
        {"Model": "XGBoost", **xgb_results},
    ]
)

model_results

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Dummy Baseline,0.790675,0.000000,0.000000,0.000000,NaN,NaN
1,Logistic Regression,0.798079,0.540646,0.235249,0.327844,0.811328,0.472359
2,Random Forest,0.760832,0.449056,0.628360,0.523788,0.830193,0.476627
3,Extra Trees,0.796042,0.531344,0.217331,0.308485,0.828035,0.478166
4,HistGradientBoosting,0.799437,0.558130,0.200958,0.295514,0.837569,0.502270
5,XGBoost,0.800246,0.561208,0.209608,0.305218,0.838024,0.502038


## Cross-Validation

Stratified K-fold cross-validation is used to assess the stability of candidate model performance across multiple training and validation folds.

Stratification preserves the churn-like class distribution in every fold.

Preprocessing remains inside each model pipeline so that imputation and scaling are fitted independently within each training fold, preventing cross-validation leakage.

The held-out test set remains untouched.


In [38]:
from sklearn.model_selection import StratifiedKFold, cross_validate

In [39]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [40]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

In [41]:
lr_cv = cross_validate(
    logistic_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False,
)

In [42]:
lr_cv_results = {
    metric: {
        "mean": lr_cv[f"test_{metric}"].mean(),
        "std": lr_cv[f"test_{metric}"].std(),
    }
    for metric in scoring
}

pd.DataFrame(lr_cv_results).T

,mean,std
accuracy,0.797657,0.001841
precision,0.539447,0.010504
recall,0.228610,0.004244
f1,0.321120,0.005795
roc_auc,0.810931,0.002893
pr_auc,0.469627,0.007124


In [43]:
rf_cv = cross_validate(
    random_forest_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False,
)

rf_cv_results = {
    metric: {
        "mean": rf_cv[f"test_{metric}"].mean(),
        "std": rf_cv[f"test_{metric}"].std(),
    }
    for metric in scoring
}

pd.DataFrame(rf_cv_results).T

,mean,std
accuracy,0.760249,0.002971
precision,0.447946,0.004764
recall,0.624830,0.003715
f1,0.521800,0.004371
roc_auc,0.828959,0.003237
pr_auc,0.474902,0.009833


In [44]:
et_cv = cross_validate(
    extra_trees_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False,
)

In [45]:
et_cv_results = {
    metric: {
        "mean": et_cv[f"test_{metric}"].mean(),
        "std": et_cv[f"test_{metric}"].std(),
    }
    for metric in scoring
}

pd.DataFrame(et_cv_results).T

,mean,std
accuracy,0.793707,0.002997
precision,0.518688,0.018397
recall,0.204283,0.004669
f1,0.293108,0.007670
roc_auc,0.826581,0.003296
pr_auc,0.471902,0.008844


In [46]:
hgb_cv = cross_validate(
    hist_gb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False,
)

In [47]:
hgb_cv_results = {
    metric: {
        "mean": hgb_cv[f"test_{metric}"].mean(),
        "std": hgb_cv[f"test_{metric}"].std(),
    }
    for metric in scoring
}

pd.DataFrame(hgb_cv_results).T

,mean,std
accuracy,0.799243,0.002151
precision,0.559857,0.016183
recall,0.192665,0.005101
f1,0.286613,0.006502
roc_auc,0.835562,0.003055
pr_auc,0.496628,0.008140


In [48]:
hgb_fold_results = pd.DataFrame({
    metric: hgb_cv[f"test_{metric}"]
    for metric in scoring
})

hgb_fold_results

,accuracy,precision,recall,f1,roc_auc,pr_auc
0,0.802051,0.578394,0.200265,0.297517,0.838425,0.503949
1,0.797132,0.542675,0.196757,0.288803,0.832835,0.487522
2,0.801670,0.580466,0.189806,0.286071,0.839031,0.507624
3,0.797547,0.547358,0.190303,0.282417,0.836260,0.495991
4,0.797817,0.550391,0.186197,0.278259,0.831260,0.488057


In [49]:
xgb_cv = cross_validate(
    xgb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False,
)

In [52]:
xgb_cv_results = {
    metric: {
        "mean": xgb_cv[f"test_{metric}"].mean(),
        "std": xgb_cv[f"test_{metric}"].std(),
    }
    for metric in scoring
}

pd.DataFrame(xgb_cv_results).T

,mean,std
accuracy,0.799333,0.001891
precision,0.555920,0.013460
recall,0.206765,0.001525
f1,0.301392,0.002979
roc_auc,0.835863,0.003143
pr_auc,0.497248,0.008544


In [53]:
xgb_fold_results = pd.DataFrame({
    metric: xgb_cv[f"test_{metric}"]
    for metric in scoring
})

xgb_fold_results

,accuracy,precision,recall,f1,roc_auc,pr_auc
0,0.800873,0.565919,0.208871,0.305126,0.838799,0.505123
1,0.796889,0.539301,0.204369,0.296412,0.833684,0.487746
2,0.802120,0.576656,0.206023,0.303584,0.839392,0.508058
3,0.798379,0.548968,0.206851,0.300481,0.836398,0.497834
4,0.798406,0.548754,0.207713,0.301357,0.831039,0.487477


In [54]:
cv_results = pd.DataFrame({
    "Logistic Regression": {
        metric: f"{lr_cv_results[metric]['mean']:.4f} ± {lr_cv_results[metric]['std']:.4f}"
        for metric in scoring
    },
    "Random Forest": {
        metric: f"{rf_cv_results[metric]['mean']:.4f} ± {rf_cv_results[metric]['std']:.4f}"
        for metric in scoring
    },
    "Extra Trees": {
        metric: f"{et_cv_results[metric]['mean']:.4f} ± {et_cv_results[metric]['std']:.4f}"
        for metric in scoring
    },
    "HistGradientBoosting": {
        metric: f"{hgb_cv_results[metric]['mean']:.4f} ± {hgb_cv_results[metric]['std']:.4f}"
        for metric in scoring
    },
    "XGBoost": {
        metric: f"{xgb_cv_results[metric]['mean']:.4f} ± {xgb_cv_results[metric]['std']:.4f}"
        for metric in scoring
    },
})

cv_results

,Logistic Regression,Random Forest,Extra Trees,HistGradientBoosting,XGBoost
accuracy,0.7977 ± 0.0018,0.7602 ± 0.0030,0.7937 ± 0.0030,0.7992 ± 0.0022,0.7993 ± 0.0019
precision,0.5394 ± 0.0105,0.4479 ± 0.0048,0.5187 ± 0.0184,0.5599 ± 0.0162,0.5559 ± 0.0135
recall,0.2286 ± 0.0042,0.6248 ± 0.0037,0.2043 ± 0.0047,0.1927 ± 0.0051,0.2068 ± 0.0015
f1,0.3211 ± 0.0058,0.5218 ± 0.0044,0.2931 ± 0.0077,0.2866 ± 0.0065,0.3014 ± 0.0030
roc_auc,0.8109 ± 0.0029,0.8290 ± 0.0032,0.8266 ± 0.0033,0.8356 ± 0.0031,0.8359 ± 0.0031
pr_auc,0.4696 ± 0.0071,0.4749 ± 0.0098,0.4719 ± 0.0088,0.4966 ± 0.0081,0.4972 ± 0.0085
